# 🏦 PySpark MLlib — Binary Classification: Bank Marketing
  ## Industry Pipeline v2 · Logistic Regression · Decision Tree · Random Forest · GBT

  > **Goal:** Predict whether a bank client subscribes to a term deposit (`deposit` = yes/no).

  ---
  ### ✅ All 13 Bugs Fixed

  | # | Bug | Fix |
  |---|-----|-----|
  | 1 | `pip install` missing `!` | Added `!` |
  | 2 | **No Java install → JAVA_GATEWAY_EXITED crash** | `apt-get install default-jdk` + auto JAVA_HOME |
  | 3 | No dataset download → `/content/bank.csv` missing | Auto-downloads from public URL |
  | 4 | Cell 4: garbage scratch notes (`1,2,3--4`) | Removed, replaced with markdown |
  | 5 | `VectorAssembler` **truncated** — missing `outputCol='features'` | Fixed |
  | 6 | ROC curve **xlabel/ylabel SWAPPED** | Fixed: FPR→X, TPR→Y |
  | 7 | `import matplotlib` placed mid-notebook | Moved to top imports |
  | 8 | `train.show()` dumps raw vector data | Removed, replaced with EDA |
  | 9 | Cell 23: `MulticlassClassificationEvaluator` unused | Now used: Accuracy/F1/Precision/Recall |
  | 10 | Only AUC reported — no Accuracy/F1/Precision/Recall | All 4 metrics added |
  | 11 | No confusion matrix | Added heatmap for all 4 models |
  | 12 | No model comparison | Full comparison dashboard added |
  | 13 | GBTClassifier missing | Added as 4th model |

---
## ⚙️ Step 0 — Install Java & Dependencies

> ✅ **Fix #2 (Critical):** Java must be installed *before* PySpark — without it you get `PySparkRuntimeError: JAVA_GATEWAY_EXITED`.

In [ ]:
import subprocess, sys, os

  # 1️⃣ Install OpenJDK 11 — required by PySpark (was MISSING in original)
  print("📦 [1/3] Installing Java 11 ...")
  subprocess.run(['apt-get','install','-y','-q','default-jdk'], capture_output=True)

  # 2️⃣ Install Python packages
  # ✅ Fix #1: Added '!' equivalent — original had bare 'pip install pyspark' with no '!'
  print("📦 [2/3] Installing PySpark + plotting libraries ...")
  subprocess.check_call([sys.executable,'-m','pip','install','pyspark','matplotlib','seaborn','-q'])

  # 3️⃣ Auto-detect JAVA_HOME (works on Colab, local Jupyter, JupyterHub)
  def _find_java_home():
      candidates = [
          '/usr/lib/jvm/java-11-openjdk-amd64',
          '/usr/lib/jvm/java-17-openjdk-amd64',
          '/usr/lib/jvm/java-21-openjdk-amd64',
          '/usr/lib/jvm/java-8-openjdk-amd64',
          '/usr/lib/jvm/default-java',
      ]
      for p in candidates:
          if os.path.isdir(p): return p
      try:
          java_bin = subprocess.check_output(['readlink','-f','/usr/bin/java'],text=True).strip()
          return os.path.dirname(os.path.dirname(java_bin))
      except Exception:
          return '/usr/lib/jvm/default-java'

  java_home = _find_java_home()
  os.environ['JAVA_HOME'] = java_home
  print(f"📦 [3/3] JAVA_HOME → {java_home}")

  try:
      ver = subprocess.check_output(['java','-version'],stderr=subprocess.STDOUT,text=True).splitlines()[0]
      print(f"\n✅ Java : {ver}")
  except Exception as e:
      print(f"\n❌ Java not accessible: {e}")
      print("   Restart runtime and re-run this cell.")
  print("✅ All dependencies ready")

---
## 📦 Step 1 — Imports

> ✅ Fix #7: All imports at the top — `import matplotlib` was placed mid-notebook in the original.

In [ ]:
import os, urllib.request
  import numpy  as np
  import pandas as pd
  import matplotlib.pyplot  as plt
  import matplotlib.patches as mpatches
  import seaborn as sns

  from pyspark.sql import SparkSession
  from pyspark.sql.functions import col
  from pyspark.ml import Pipeline
  from pyspark.ml.feature    import StringIndexer, OneHotEncoder, VectorAssembler
  from pyspark.ml.classification import (
      LogisticRegression,
      DecisionTreeClassifier,
      RandomForestClassifier,
      GBTClassifier,                        # ✅ Fix #13: added
  )
  from pyspark.ml.evaluation import (
      BinaryClassificationEvaluator,
      MulticlassClassificationEvaluator,    # ✅ Fix #9: now actually used
  )
  print("✅ Imports complete")

---
## 🚀 Step 2 — SparkSession

In [ ]:
# Guard: set JAVA_HOME if Step 0 was skipped
  if not os.environ.get('JAVA_HOME'):
      for _p in ['/usr/lib/jvm/java-11-openjdk-amd64','/usr/lib/jvm/default-java']:
          if os.path.isdir(_p): os.environ['JAVA_HOME'] = _p; break

  spark = SparkSession.builder \
      .appName("BankMarketing_Classification_v2") \
      .master("local[*]") \
      .config("spark.ui.port",               "4050") \
      .config("spark.executor.memory",       "1g") \
      .config("spark.driver.memory",         "1g") \
      .config("spark.sql.shuffle.partitions","8") \
      .config("spark.ui.showConsoleProgress","false") \
      .getOrCreate()

  spark.sparkContext.setLogLevel("ERROR")
  print(f"✅ SparkSession ready  |  Spark {spark.version}  |  {spark.sparkContext.defaultParallelism} cores")

---
## 📂 Step 3 — Load Bank Marketing Dataset

> ✅ Fix #3: Auto-downloads dataset. Original had no download step — `/content/bank.csv` was always missing.

In [ ]:
LOCAL_PATH = "/content/bank.csv"

  URLS = [
      "https://raw.githubusercontent.com/dsrscientist/dataset1/master/bank.csv",
      "https://raw.githubusercontent.com/madmashup/targeted-marketing-predictive-engine/master/banking.csv",
  ]
  downloaded = False
  for url in URLS:
      try:
          urllib.request.urlretrieve(url, LOCAL_PATH)
          _c = pd.read_csv(LOCAL_PATH)
          if 'deposit' in _c.columns and len(_c) > 100:
              print(f"✅ Dataset downloaded  ({len(_c):,} rows)")
              downloaded = True; break
      except Exception as e:
          print(f"   URL failed: {e}")

  if not downloaded:
      print("⚠️  Generating synthetic Bank Marketing dataset ...")
      np.random.seed(42); n = 4521
      JOBS      = ['admin.','blue-collar','entrepreneur','housemaid','management',
                   'retired','self-employed','services','student','technician','unemployed','unknown']
      MARITALS  = ['divorced','married','single']
      EDUS      = ['primary','secondary','tertiary','unknown']
      CONTACTS  = ['cellular','telephone','unknown']
      POUTCOMES = ['failure','other','success','unknown']
      age      = np.random.randint(18, 95, n)
      balance  = np.random.normal(1362, 3000, n).astype(int)
      duration = np.random.exponential(300, n).astype(int).clip(1,3600)
      campaign = np.random.randint(1, 30, n)
      pdays    = np.where(np.random.rand(n)<0.82, -1, np.random.randint(1,400,n))
      previous = np.where(pdays==-1, 0, np.random.randint(0,10,n))
      dep_prob = (0.3+0.25*(duration>400)+0.10*(balance>1000)
                  +0.05*(age<30)+0.05*(age>60)-0.05*(campaign>5)).clip(0.05,0.95)
      deposit  = np.where(np.random.rand(n)<dep_prob,'yes','no')
      synthetic = pd.DataFrame({
          'age':age,'job':np.random.choice(JOBS,n),
          'marital':np.random.choice(MARITALS,n,p=[0.13,0.60,0.27]),
          'education':np.random.choice(EDUS,n,p=[0.15,0.52,0.29,0.04]),
          'default':np.random.choice(['yes','no'],n,p=[0.02,0.98]),
          'balance':balance,
          'housing':np.random.choice(['yes','no'],n,p=[0.56,0.44]),
          'loan':np.random.choice(['yes','no'],n,p=[0.16,0.84]),
          'contact':np.random.choice(CONTACTS,n,p=[0.65,0.12,0.23]),
          'duration':duration,'campaign':campaign,'pdays':pdays,'previous':previous,
          'poutcome':np.random.choice(POUTCOMES,n,p=[0.11,0.04,0.03,0.82]),
          'deposit':deposit,
      })
      synthetic.to_csv(LOCAL_PATH, index=False)
      print(f"✅ Synthetic dataset saved ({n:,} rows × {len(synthetic.columns)} cols)")

  df = spark.read.csv(LOCAL_PATH, header=True, inferSchema=True)
  print(f"\nLoaded: {df.count():,} rows × {len(df.columns)} cols")
  df.printSchema()

---
## 🔍 Step 4 — Exploratory Data Analysis

In [ ]:
# Numeric stats
  numeric_features = [t[0] for t in df.dtypes if t[1]=='int']
  print(f"Numeric columns: {numeric_features}")
  df.select(numeric_features).describe().toPandas().set_index('summary').T

In [ ]:
# ✅ Fix #8: replaced 'train.show()' debug dump with meaningful EDA visualisations
  pdf = df.toPandas()

  fig, axes = plt.subplots(2, 3, figsize=(16,9))
  fig.patch.set_facecolor('#0d0d1a')
  fig.suptitle('Bank Marketing — EDA Dashboard', fontsize=14, fontweight='bold', color='white')

  # 1. Target balance
  ax=axes[0][0]; ax.set_facecolor('#1a1a2e')
  vc=pdf['deposit'].value_counts()
  bars=ax.bar(vc.index,vc.values,color=['#4CAF50','#F44336'],edgecolor='#333',width=0.5)
  for b in bars:
      ax.text(b.get_x()+b.get_width()/2,b.get_height()+20,f'{b.get_height():,}',
              ha='center',color='white',fontsize=11,fontweight='bold')
  ax.set_title('Target Class Balance',color='white'); ax.tick_params(colors='white')
  for s in ax.spines.values(): s.set_edgecolor('#444')

  # 2. Age by deposit
  ax=axes[0][1]; ax.set_facecolor('#1a1a2e')
  for dep,col in [('yes','#4CAF50'),('no','#F44336')]:
      ax.hist(pdf[pdf['deposit']==dep]['age'],bins=25,alpha=0.6,color=col,label=dep)
  ax.set_title('Age Distribution by Deposit',color='white')
  ax.legend(facecolor='#0d0d1a',labelcolor='white',fontsize=8)
  ax.tick_params(colors='white')
  for s in ax.spines.values(): s.set_edgecolor('#444')

  # 3. Duration by deposit
  ax=axes[0][2]; ax.set_facecolor('#1a1a2e')
  for dep,col in [('yes','#4CAF50'),('no','#F44336')]:
      ax.hist(pdf[pdf['deposit']==dep]['duration'].clip(0,1500),bins=30,alpha=0.6,color=col,label=dep)
  ax.set_title('Call Duration by Deposit',color='white')
  ax.legend(facecolor='#0d0d1a',labelcolor='white',fontsize=8)
  ax.tick_params(colors='white')
  for s in ax.spines.values(): s.set_edgecolor('#444')

  # 4. Job vs deposit rate
  ax=axes[1][0]; ax.set_facecolor('#1a1a2e')
  jr=pdf.groupby('job')['deposit'].apply(lambda x:(x=='yes').mean()).sort_values(ascending=True)
  c=plt.cm.RdYlGn(np.linspace(0.1,0.9,len(jr)))
  ax.barh(jr.index,jr.values,color=c,edgecolor='#333')
  ax.set_xlabel('Deposit Rate',color='white'); ax.set_title('Deposit Rate by Job',color='white')
  ax.tick_params(colors='white',labelsize=7.5)
  for s in ax.spines.values(): s.set_edgecolor('#444')

  # 5. Education vs deposit rate
  ax=axes[1][1]; ax.set_facecolor('#1a1a2e')
  er=pdf.groupby('education')['deposit'].apply(lambda x:(x=='yes').mean())
  bars=ax.bar(er.index,er.values,color=['#2196F3','#4CAF50','#FF9800','#9C27B0'],edgecolor='#333',width=0.5)
  for b in bars:
      ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.005,f'{b.get_height():.2f}',
              ha='center',color='white',fontsize=9)
  ax.set_title('Deposit Rate by Education',color='white'); ax.tick_params(colors='white')
  for s in ax.spines.values(): s.set_edgecolor('#444')

  # 6. Balance vs Duration
  ax=axes[1][2]; ax.set_facecolor('#1a1a2e')
  for dep,col in [('yes','#4CAF50'),('no','#F44336')]:
      s=pdf[pdf['deposit']==dep].sample(min(500,len(pdf[pdf['deposit']==dep])),random_state=42)
      ax.scatter(s['balance'].clip(-1000,5000),s['duration'].clip(0,1500),
                 alpha=0.35,s=10,color=col,label=dep)
  ax.set_xlabel('Balance',color='white'); ax.set_ylabel('Duration',color='white')
  ax.set_title('Balance vs Duration',color='white')
  ax.legend(facecolor='#0d0d1a',labelcolor='white',fontsize=8)
  ax.tick_params(colors='white')
  for s in ax.spines.values(): s.set_edgecolor('#444')

  plt.tight_layout()
  plt.savefig('eda_bank.png',dpi=120,bbox_inches='tight',facecolor=fig.get_facecolor())
  plt.show(); print("✅ EDA saved")

---
  ## 🔧 Step 5 — Feature Engineering (Pipeline)

  > ✅ Fix #4: Removed garbage scratch-note cell (`1,2,3--4` junk).  
  > ✅ Fix #5: Fixed truncated `VectorAssembler` — `outputCol='features'` was missing.

  ```
  Categorical col  →  StringIndexer  →  OneHotEncoder  ─┐
                                                          ├→ VectorAssembler → features
  Numeric cols     ──────────────────────────────────────┘
  deposit          →  StringIndexer  →  label (0/1)
  ```

In [ ]:
df = df.select('age','job','marital','education','default','balance','housing',
                 'loan','contact','duration','campaign','pdays','previous','poutcome','deposit')

  categoricalColumns = ['job','marital','education','default','housing','loan','contact','poutcome']
  numericCols        = ['age','balance','duration','campaign','pdays','previous']
  stages = []

  for c in categoricalColumns:
      si  = StringIndexer(inputCol=c, outputCol=c+'Index', handleInvalid='keep')
      enc = OneHotEncoder(inputCols=[si.getOutputCol()], outputCols=[c+'classVec'])
      stages += [si, enc]

  stages += [StringIndexer(inputCol='deposit', outputCol='label')]

  # ✅ Fix #5: VectorAssembler was CUT OFF in original — outputCol='features' was missing
  assemblerInputs = [c+'classVec' for c in categoricalColumns] + numericCols
  stages += [VectorAssembler(inputCols=assemblerInputs, outputCol='features', handleInvalid='keep')]

  print(f"✅ Pipeline: {len(stages)} stages  |  "
        f"{len(categoricalColumns)} categorical + {len(numericCols)} numeric features")

  pipeline     = Pipeline(stages=stages)
  pipelineModel= pipeline.fit(df)
  df_tr        = pipelineModel.transform(df)
  model_df     = df_tr.select(['label','features'])
  model_df.show(3)

---
## ✂️ Step 6 — Train / Test Split

In [ ]:
train, test = model_df.randomSplit([0.7, 0.3], seed=42)
  train.cache(); test.cache()
  print(f"✅ Train: {train.count():,}   Test: {test.count():,}  (cached)")

In [ ]:
def evaluate_model(preds, name):
      """AUC + Accuracy + F1 + Precision + Recall for binary classification."""
      b  = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
      mc = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction')
      auc  = b.evaluate(preds)
      acc  = mc.evaluate(preds, {mc.metricName:'accuracy'})
      f1   = mc.evaluate(preds, {mc.metricName:'f1'})
      prec = mc.evaluate(preds, {mc.metricName:'weightedPrecision'})
      rec  = mc.evaluate(preds, {mc.metricName:'weightedRecall'})
      print(f"\n{'─'*52}")
      print(f"  📊 {name}")
      print(f"  AUC       : {auc:.4f}")
      print(f"  Accuracy  : {acc:.4f}")
      print(f"  F1 Score  : {f1:.4f}")
      print(f"  Precision : {prec:.4f}")
      print(f"  Recall    : {rec:.4f}")
      print(f"{'─'*52}")
      return dict(model=name,auc=auc,accuracy=acc,f1=f1,precision=prec,recall=rec)

  def confusion_heatmap(preds, name, ax, color):
      from sklearn.metrics import confusion_matrix
      p = preds.select('label','prediction').toPandas()
      cm= confusion_matrix(p['label'],p['prediction'])
      ax.imshow(cm, cmap='Blues'); ax.set_facecolor('#1a1a2e')
      for i in range(cm.shape[0]):
          for j in range(cm.shape[1]):
              ax.text(j,i,f'{cm[i,j]:,}',ha='center',va='center',
                      fontsize=12,fontweight='bold',
                      color='white' if cm[i,j]>cm.max()/2 else 'black')
      ax.set_xticks([0,1]); ax.set_yticks([0,1])
      ax.set_xticklabels(['Pred No','Pred Yes'],color='white',fontsize=8)
      ax.set_yticklabels(['Act. No','Act. Yes'],color='white',fontsize=8,rotation=90,va='center')
      ax.set_title(name,color=color,fontsize=10,fontweight='bold')
  print("✅ Helpers ready")

---
## 📈 Step 7 — Model 1: Logistic Regression

Probabilistic linear classifier — fast, interpretable baseline.

In [ ]:
lr = LogisticRegression(
      featuresCol='features', labelCol='label',
      maxIter=100,       # ✅ raised from 10 for convergence
      regParam=0.01, elasticNetParam=0.0)
  lr_model = lr.fit(train)
  lr_preds  = lr_model.transform(test)
  lr_m      = evaluate_model(lr_preds, "Logistic Regression")
  lr_preds.select('label','prediction','probability').show(5)

In [ ]:
# ✅ Fix #6: ROC curve xlabel/ylabel were SWAPPED in original
  # Original: plt.ylabel('False Positive Rate') + plt.xlabel('True Positive Rate') ← WRONG
  # Fixed   : plt.xlabel('False Positive Rate') + plt.ylabel('True Positive Rate') ← CORRECT
  summary = lr_model.summary
  roc_pdf = summary.roc.toPandas()
  pr_pdf  = summary.pr.toPandas()

  fig, axes = plt.subplots(1, 2, figsize=(13,5))
  fig.patch.set_facecolor('#0d0d1a')
  fig.suptitle('Logistic Regression — ROC & Precision-Recall Curves',
               fontsize=12, fontweight='bold', color='white')

  ax1=axes[0]; ax1.set_facecolor('#1a1a2e')
  ax1.plot(roc_pdf['FPR'],roc_pdf['TPR'],color='#2196F3',lw=2,
           label=f"AUC = {summary.areaUnderROC:.3f}")
  ax1.plot([0,1],[0,1],'w--',lw=1,label='Random')
  ax1.set_xlabel('False Positive Rate',color='white')   # ✅ FIXED (was "True Positive Rate")
  ax1.set_ylabel('True Positive Rate', color='white')   # ✅ FIXED (was "False Positive Rate")
  ax1.set_title('ROC Curve',color='white')
  ax1.legend(facecolor='#0d0d1a',labelcolor='white',fontsize=9)
  ax1.tick_params(colors='white')
  for s in ax1.spines.values(): s.set_edgecolor('#444')

  ax2=axes[1]; ax2.set_facecolor('#1a1a2e')
  ax2.plot(pr_pdf['recall'],pr_pdf['precision'],color='#4CAF50',lw=2)
  ax2.set_xlabel('Recall',color='white'); ax2.set_ylabel('Precision',color='white')
  ax2.set_title('Precision-Recall Curve',color='white')
  ax2.tick_params(colors='white')
  for s in ax2.spines.values(): s.set_edgecolor('#444')

  plt.tight_layout()
  plt.savefig('lr_curves.png',dpi=120,bbox_inches='tight',facecolor=fig.get_facecolor())
  plt.show()

---
## 🌳 Step 8 — Model 2: Decision Tree Classifier

In [ ]:
dt = DecisionTreeClassifier(
      featuresCol='features', labelCol='label',
      maxDepth=5, minInstancesPerNode=10, seed=42)
  dt_model = dt.fit(train)
  dt_preds = dt_model.transform(test)
  dt_m     = evaluate_model(dt_preds, "Decision Tree")
  dt_preds.select('label','prediction','probability').show(5)

---
## 🌲 Step 9 — Model 3: Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(
      featuresCol='features', labelCol='label',
      numTrees=100, maxDepth=5, seed=42)
  rf_model = rf.fit(train)
  rf_preds = rf_model.transform(test)
  rf_m     = evaluate_model(rf_preds, "Random Forest")

---
## 🚀 Step 10 — Model 4: Gradient-Boosted Tree Classifier

> ✅ Fix #13: Added GBTClassifier — missing entirely from original notebook.

In [ ]:
gbt = GBTClassifier(
      featuresCol='features', labelCol='label',
      maxIter=50, maxDepth=5, stepSize=0.1,
      subsamplingRate=0.8, seed=42)
  gbt_model = gbt.fit(train)
  gbt_preds = gbt_model.transform(test)
  gbt_m     = evaluate_model(gbt_preds, "GBT Classifier")

---
## 📊 Step 11 — Confusion Matrices (All 4 Models)

> ✅ Fix #11: Added — not present in original.

In [ ]:
try:
      from sklearn.metrics import confusion_matrix
      all_preds = [(lr_preds,'Logistic Regression','#2196F3'),
                   (dt_preds,'Decision Tree',       '#FF9800'),
                   (rf_preds,'Random Forest',        '#4CAF50'),
                   (gbt_preds,'GBT Classifier',      '#E91E63')]

      fig, axes = plt.subplots(1,4,figsize=(18,4))
      fig.patch.set_facecolor('#0d0d1a')
      fig.suptitle('Confusion Matrices — All 4 Models',fontsize=13,fontweight='bold',color='white')
      for ax,(preds,name,col) in zip(axes,all_preds):
          confusion_heatmap(preds,name,ax,col)
      plt.tight_layout()
      plt.savefig('confusion_matrices.png',dpi=120,bbox_inches='tight',facecolor=fig.get_facecolor())
      plt.show()
  except ImportError:
      print("sklearn not available — run: !pip install scikit-learn -q")

---
## 🔑 Step 12 — Feature Importance (RF & GBT)

> ✅ Fix #9: properly computed and visualised — original had evaluator created but never used.

In [ ]:
# Build short feature names post-encoding
  feature_names = ([f'{c}_{i}' for c in ['job','marital','education','default',
                                           'housing','loan','contact','poutcome']
                    for i in range(5)] + ['age','balance','duration','campaign','pdays','previous'])
  n_feats = rf_model.featureImportances.size
  feature_names = (feature_names + [f'feat_{i}' for i in range(n_feats)])[:n_feats]

  def plot_fi(imp_vec, title, color, ax, top=15):
      fi = pd.DataFrame({'feature':feature_names,'importance':imp_vec.toArray()})
      fi = fi.nlargest(top,'importance').sort_values('importance',ascending=True)
      c  = plt.cm.plasma(np.linspace(0.2,0.9,len(fi)))
      bars = ax.barh(fi['feature'],fi['importance'],color=c,edgecolor='#333')
      for b,v in zip(bars,fi['importance']):
          ax.text(b.get_width()+0.001,b.get_y()+b.get_height()/2,
                  f'{v:.3f}',va='center',fontsize=7.5,color='white')
      ax.set_facecolor('#1a1a2e'); ax.tick_params(colors='white',labelsize=8)
      ax.set_xlabel('Importance',color='white')
      ax.set_title(title,color=color,fontsize=10,fontweight='bold')
      for s in ax.spines.values(): s.set_edgecolor('#444')

  fig, axes = plt.subplots(1,2,figsize=(16,7))
  fig.patch.set_facecolor('#0d0d1a')
  fig.suptitle('Top-15 Feature Importances',fontsize=13,fontweight='bold',color='white')
  plot_fi(rf_model.featureImportances, 'Random Forest','#4CAF50',axes[0])
  plot_fi(gbt_model.featureImportances,'GBT Classifier','#E91E63',axes[1])
  plt.tight_layout()
  plt.savefig('feature_importance.png',dpi=120,bbox_inches='tight',facecolor=fig.get_facecolor())
  plt.show()

---
## 🏆 Step 13 — Model Comparison Dashboard

> ✅ Fix #12: Added — missing from original.

In [ ]:
all_m  = [lr_m, dt_m, rf_m, gbt_m]
  metrics= [('auc','AUC ↑',True),('accuracy','Accuracy ↑',True),
            ('f1','F1 Score ↑',True),('precision','Precision ↑',True),('recall','Recall ↑',True)]
  cols   = ['#2196F3','#FF9800','#4CAF50','#E91E63']

  fig, axes = plt.subplots(1,5,figsize=(22,5))
  fig.patch.set_facecolor('#0d0d1a')
  fig.suptitle('🏆 Model Comparison — Bank Marketing Classification',
               fontsize=13,fontweight='bold',color='white')

  for ax,(key,label,higher) in zip(axes,metrics):
      ax.set_facecolor('#1a1a2e')
      vals = [m[key] for m in all_m]
      best = vals.index(max(vals) if higher else min(vals))
      bars = ax.bar(range(4), vals, color=cols, edgecolor='#333', width=0.6)
      bars[best].set_edgecolor('gold'); bars[best].set_linewidth(3)
      for b,v in zip(bars,vals):
          ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.003,
                  f'{v:.3f}',ha='center',fontsize=9,color='white',fontweight='bold')
      ax.text(bars[best].get_x()+bars[best].get_width()/2,
              bars[best].get_height()*0.45,'🏆',ha='center',fontsize=14)
      ax.set_title(label,color='white',fontsize=10)
      ax.set_xticks(range(4))
      ax.set_xticklabels(['LR','DT','RF','GBT'],color='white',fontsize=9)
      ax.tick_params(colors='white',axis='y',labelsize=8)
      ypad=0.04; ax.set_ylim(max(0,min(vals)-ypad),min(1.05,max(vals)+ypad*2))
      for s in ax.spines.values(): s.set_edgecolor('#444')

  plt.tight_layout()
  plt.savefig('model_comparison.png',dpi=120,bbox_inches='tight',facecolor=fig.get_facecolor())
  plt.show()

In [ ]:
print("="*70)
  print("  FINAL MODEL COMPARISON — BANK MARKETING CLASSIFICATION")
  print("="*70)
  print(f"{'Model':<22} {'AUC':>7} {'Acc':>7} {'F1':>7} {'Prec':>7} {'Rec':>7}")
  print("─"*70)
  for m in all_m:
      best = lambda k: "⭐" if m[k]==max(x[k] for x in all_m) else "  "
      print(f"{m['model']:<22} {m['auc']:>6.4f}{best('auc')} "
            f"{m['accuracy']:>6.4f}{best('accuracy')} {m['f1']:>6.4f}{best('f1')} "
            f"{m['precision']:>6.4f}{best('precision')} {m['recall']:>6.4f}{best('recall')}")
  print("="*70)
  best_m = max(all_m, key=lambda x: x['auc'])
  print(f"\n✅ Best model by AUC: {best_m['model']}  (AUC={best_m['auc']:.4f})")

---
## 🔚 Step 14 — Clean Shutdown

In [ ]:
train.unpersist(); test.unpersist()
  spark.stop()
  print("✅ SparkSession stopped cleanly.")

---
  ## 📖 MLlib Classification Cheat Sheet

  | API | Key Params | Notes |
  |-----|-----------|-------|
  | `LogisticRegression` | `maxIter`, `regParam`, `elasticNetParam` | Interpretable linear baseline |
  | `DecisionTreeClassifier` | `maxDepth`, `minInstancesPerNode`, `seed` | Prone to overfit at high depth |
  | `RandomForestClassifier` | `numTrees`, `maxDepth`, `seed` | Strong ensemble; robust |
  | `GBTClassifier` | `maxIter`, `stepSize`, `subsamplingRate` | **Binary only**; usually best |
  | `BinaryClassificationEvaluator` | `metricName='areaUnderROC'` | Standard AUC metric |
  | `MulticlassClassificationEvaluator` | `accuracy/f1/weightedPrecision/weightedRecall` | All other classification metrics |
  | `StringIndexer` | `inputCol`, `outputCol`, `handleInvalid='keep'` | String → numeric index |
  | `OneHotEncoder` | `inputCols`, `outputCols` | Index → sparse one-hot vector |
  | `VectorAssembler` | `inputCols`, `outputCol='features'` | Pack all into one feature vector |